# Atomic-GDN2: actual full-LM coordinate mechanism audit

**Akseleu Omirbay.** Следующий уровень после контролируемого Decimal-примера: реальная модель **34 873 952 параметра, B8, L2048**, исходная координата **block_6/q_conv_w[236,2]**. Seed 0 — исходный проблемный случай; seeds 1 и 2 — заранее заданные контроли. **OLD / JAX_REF / NEW atomic_gdn2**, zero damping, исходная зафиксированная версия ядра.

## Запуск
Новая Colab среда **TPU v5e-1**, затем **Run all**. Не запускайте одновременно другой процесс в этой VM. Токены, Drive и GitHub не нужны: проверенный снимок исходников встроен. Это диагностика, не новый speed benchmark. Шесть последовательных процессов (control/probe × три реализации), каждый с тремя сидами; компиляции полной модели могут занять значительное время. Один процесс владеет TPU в каждый момент. В конце скачивается один ZIP. Все ошибки и частичные результаты сохраняются.

## Что именно разделяем
1. **Неизменённая модель:** заново получаем градиент исходной координаты и global norm.
2. **Нулевой зонд:** сохраняем реальный вход этой свёртки и производную loss по её выходу для выбранного канала. Отдельно измеряем влияние инструментации на loss/градиент. Нулевое значение зонда не гарантирует прежний порядок операций XLA.
3. **Точный локальный расчёт:** градиент веса есть сумма произведений сдвинутого входа и cotangent. Все 16 384 слагаемых сохраняются. Их сумма пересчитывается **точной рациональной арифметикой**, без FP32/FP64 округления. Это эталон для сохранённых факторов, а не точный градиент всей нейросети.
4. **Разделение причин:** сравниваем TPU gradient с точной суммой; отдельно заменяем только вход или только cotangent между OLD/JAX_REF/NEW. Точное алгебраическое разложение показывает вклад различий во входе и в обратном сигнале. FP32/FP64/fsum проверяют чувствительность самого последнего суммирования.
5. **Adam:** epsilon sweep на измеренных и локально пересчитанных градиентах с исходным global clipping. Adaptive update и weight decay записываются отдельно.

## Как читать результат
- Если TPU gradient отличается от точной суммы сохранённых факторов — локализован численный остаток последней свёртки/редукции.
- Если точные суммы OLD и WY уже различаются, часть расхождения пришла **до этой редукции**. Таблицы показывают, через вход или cotangent. Это нельзя выдавать за доказательство того, что виновато только суммирование в свёртке.
- Большая cancellation ratio показывает чувствительность суммы. Она не доказывает истинный градиент 0 и не определяет, какой upstream kernel точнее.
- Контроль probe/control обязателен: инструментация может изменить чувствительную координату. Значительное изменение ограничивает перенос вывода на исходное исполнение.

Этот эксперимент вместе с предыдущим независимым Decimal-примером углубляет проверку механизма. Он не обещает заранее исключить все баги Pallas, доказать отсутствие влияния на качество или получить заранее выбранный ответ. Все девять path/seed случаев и пропуски идут в публикационные таблицы.


In [ ]:
import sys,subprocess,pathlib,zipfile,hashlib,json,datetime,base64,zlib,os
RUN_ID=datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
WORK=pathlib.Path('/content')/('atomic-actual-coordinate-'+RUN_ID);WORK.mkdir()
OUT=WORK/'results';OUT.mkdir();source=WORK/'source';source.mkdir()
subprocess.check_call([sys.executable,'-m','pip','install','-q','virtualenv'])
subprocess.check_call([sys.executable,'-m','virtualenv',str(WORK/'venv')])
PY=str(WORK/'venv/bin/python')
PACKAGES=['jax[tpu]==0.11.2','jaxlib==0.11.2','libtpu==0.0.48.1','numpy==2.1.3','scipy==1.16.3','flax==0.12.9','optax==0.2.8','ml_dtypes==0.6.0','opt_einsum==3.4.0']
subprocess.check_call([PY,'-m','pip','install','-q',*PACKAGES])
(OUT/'pip-freeze.txt').write_text(subprocess.check_output([PY,'-m','pip','freeze'],text=True))


In [ ]:
BLOB='...'
DERIVATION={'parent_archive_sha256': 'beec2c72c072e1a11efc82217585e04a409fd1a62a698dd83291b8b7362678cb', 'pinned_revision': '11616acd8a061768a38feb12c58e5e329952f5d5', 'hashes': {'atomic_gdn2/__init__.py': '7fcd8e1f64923cc5f4bc26aacda65a5ab797ee85c9e8a158d0ba9c8ee0f18142', 'atomic_gdn2/bwd/__init__.py': 'ebcdf69182f0a89a2227b1256cf3d9c011cb5ee04f486be9cf79a93c1e24a24a', 'atomic_gdn2/bwd/b4.py': '46f16bf7106a503c4b74536eeb4cb67cdf637f0eb738e15524cc5d39a5abd20e', 'atomic_gdn2/bwd/mega.py': '7af521b333bccdddb8e8e5e492631c7cfa786e9af77e4730c668d469b1dd0f17', 'atomic_gdn2/config.py': '2c84e4e2868a5ef233392a02b188045d87abeb16b7ac87bc1e18ed563c17e455', 'atomic_gdn2/domain.py': '34e88f51c9444b82b44b083a15f9dec46450c90b0beb9c49b992363cdd924ced', 'atomic_gdn2/fwd/__init__.py': 'f2b5315ff0c53b69b6fdc1ac35b39f5d3f0539ffa3a0cefd1195c8b49702889e', 'atomic_gdn2/fwd/cd_slim.py': 'd6dc8c1263ee9b640dc6e824dc411b46c9b771045ed5022e5a33ae2a8307b15b', 'atomic_gdn2/fwd/scores.py': '23d9db95633fd04bfae1ac162537bafe4ca5c180d74336da82eb948c69b7d728', 'atomic_gdn2/fwd/solve.py': '19b91ebbef82c2cd9d443d41b7a75ef6d7e82bbbd0dc925814c7a3d88413ede8', 'atomic_gdn2/layer.py': '6b18e1b925faa7664bc2fffcab4efab7c3b059b06f10aa7887518ed86566b112', 'atomic_gdn2/layout.py': '0df9dedbacdfab9c354ca6c95aba09e7df495d800388b5f2d9ddb5dc11f6769c', 'atomic_gdn2/pipeline.py': 'e7169d3f6d3ce2ed72fd14b25db1068aed2e8dce01c121a96ce35798e76297cf', 'atomic_gdn2/precision.py': '4bfd3485e20f9b1091a76f8bbcb2a98ef07fda3ba36151e18cda26350b140cc4', 'atomic_gdn2/reference/__init__.py': '7db881db3e8b52992ce3c9f7ab1568992aa649f3637c622ff76c9fec9f83106d', 'atomic_gdn2/reference/ref.py': '736cfb4a82844a3eb6cdd9347f0c9315f69881cc6b23b44ef4259042de447eec', 'atomic_gdn2/training/__init__.py': '7ccc6a2897ada0293f7d5c1c04da5568910148534a425e045559007fb04a632f', 'atomic_gdn2/training/canary.py': 'f634a11102602076ce12b17da3688fc7280d0f04dcb5848121b8ab086ed2a154', 'atomic_gdn2/training/data.py': '2622b8f048baa214559fc888e51d70146d6ed3509eade4fecd437a4328da8c79', 'atomic_gdn2/training/model.py': '2075623b09a2462ab24e62dc6acd84926ee505b4b797dfb1f459a827e1a3dfd7', 'atomic_gdn2/training/train.py': '00749c871fb6a9d8bbe0f436d456c2a65350de237f2f8d00de59b972e64fc3df', 'atomic_ops/__init__.py': 'cdafb768f2f4238253359e3b8041bf0106ae02575dfd1b4727ad90fd6c875896', 'atomic_ops/configs.py': '415907bf504a544ebf206890022a8b5518bad84862617acb627a1693d2112e61', 'atomic_ops/fallback.py': '0d7e5ec2403f0191956455c067a1acbb9a05f9aa35eb8f5f7ab0ae0a20f4a059', 'atomic_ops/gdn2_bwd.py': '9467914c55bc1d4d86c62af960e3827dc1e028a918df0ab5bd85bdf508774936', 'atomic_ops/gdn2_fwd.py': 'ba8926278a788063febae3d94574c665f22a3d9feb00bc46351f148f6e39a327', 'atomic_ops/gdn2_pipeline.py': '6a0c7e59c87c7615e296483145ade67caa1d1dd1ebb63c4e3c9b1212b9021fc5', 'atomic_ops/reference.py': 'ee7ecc97a2ef474e29219256646ee7d118045cc798798aaed6851fd0d246ad3d', 'atomic_ops/utils.py': '69c6bc145de9e2448a86b000c92b7724b6c830bd2f5e5b63b3658af254f876a2', 'coordinate_bench/candidate_common.py': '5b723232a5ddec53890d5830f748b2202f4cda92461abfe6393242a946e4c437', 'coordinate_bench/candidate_e2e.py': 'db9014e14b7ebefd052a0ca74a680f95b5a4bafdeaaf38afea48f2e170d54bb9', 'coordinate_bench/isolated_e2e.py': 'a58491fc3bacb70a64d2569458887b679e85a1fd230d7ad6078c878ad2a11016', 'coordinate_bench/corrected_original_lm.py': '8714d76fc9d308f7f1f14585e9a27441a5c7b73ae77217b7b0d90719e24cd1b2', 'coordinate_bench/coordinate_probe_model.py': '913f8d005103cfcd81d3e1282458b3ee03ca8f474ca99a2b3b6e2f999d8ceab9', 'coordinate_bench/actual_coordinate_worker.py': 'df5aa266fe01906b470f6776f7912f884387785a9db58fdc59d8f20a1d16e326', 'coordinate_bench/actual_coordinate_analysis.py': '88f5e0029d01bd14a961922376ffe525ec362946e852afd98d31ecf3381da79f'}, 'probe_transformation': ['Original LM source retained alongside derived version.', 'Add optional probe argument to block/LM. At block_6 q convolution, sow selected input channel and add probe to its preactivation.', 'No kernel, parameter initialization, loss or model architecture replacement.', 'OLD block-remat retained in uninstrumented and probed runs. Zero probe can change XLA arithmetic: separate control required.']}
PAYLOAD=json.loads(zlib.decompress(base64.b64decode(BLOB)))
for n,s in PAYLOAD.items():
 assert not pathlib.PurePosixPath(n).is_absolute() and '..' not in pathlib.PurePosixPath(n).parts
 p=source/n;p.parent.mkdir(parents=True,exist_ok=True);p.write_text(s,encoding='utf-8')
 assert hashlib.sha256(p.read_bytes()).hexdigest()==DERIVATION['hashes'][n],n
(OUT/'source-derivation.json').write_text(json.dumps(DERIVATION,indent=2))
bench=source/'coordinate_bench'
print('PINNED_SOURCE_HASHES_VERIFIED',len(PAYLOAD))


In [ ]:
import queue,threading,time
STATUS={'status':'running','cases':[],'purpose':'Actual-coordinate mechanism audit; not timing'}
DEST=WORK.parent/('Atomic-gdn2-actual-coordinate-results-'+RUN_ID+'.zip')
def collect():
 (OUT/'status.json').write_text(json.dumps(STATUS,indent=2))
 manifest={};tmp=DEST.with_suffix('.tmp')
 with zipfile.ZipFile(tmp,'w',zipfile.ZIP_DEFLATED) as z:
  for p in OUT.rglob('*'):
   if p.is_file():
    n='results/'+str(p.relative_to(OUT));z.write(p,n);manifest[n]=hashlib.sha256(p.read_bytes()).hexdigest()
  for n,s in PAYLOAD.items():
   key='executed-source/'+n;z.writestr(key,s);manifest[key]=hashlib.sha256(s.encode()).hexdigest()
  z.writestr('manifest.json',json.dumps({'files':manifest},indent=2))
 tmp.replace(DEST)
def execute(label,command,env=None):
 q=queue.Queue();print('START',label,flush=True)
 p=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,env=env)
 def read():
  try:
   for line in p.stdout:q.put(line)
  finally:q.put(None)
 t=threading.Thread(target=read,daemon=True);t.start()
 try:
  with (OUT/(label+'.log')).open('w') as log:
   while True:
    try:line=q.get(timeout=25)
    except queue.Empty:print('WORKING',label,'(compilation/computation)',flush=True);continue
    if line is None:break
    print(line,end='',flush=True);log.write(line);log.flush()
  code=p.wait()
 except BaseException:
  if p.poll() is None:p.terminate();p.wait()
  raise
 STATUS['cases'].append({'case':label,'exit_code':code});collect();return code
try:
 for phase in ['control','probe']:
  for path in ['NEW','OLD','JAX_REF']:
   label=path+'-'+phase
   execute(label,[PY,'-u',str(bench/'actual_coordinate_worker.py'),'--path',path,'--phase',phase,'--output',str(OUT/(label+'.json'))])
 STATUS['status']='executions_finished'
except BaseException as e:
 STATUS.update(status='interrupted_or_error',detail=repr(e))
finally:collect()


In [ ]:
code=execute('analysis',[PY,'-u',str(bench/'actual_coordinate_analysis.py'),str(OUT)],env=dict(os.environ,JAX_PLATFORMS='cpu'))
STATUS['analysis_exit_code']=code
p=OUT/'coordinate-analysis.json'
if p.exists():
 r=json.loads(p.read_text());STATUS['analysis_status']=r['status']
 print('ANALYSIS',r['status'],'control/probe pairs',r['complete_control_probe_seed_pairs'],'of',r['expected_control_probe_seed_pairs'])
 for row in r['coordinate_cases']:print(row)
 print('Read instrumentation-control.csv and difference-decomposition.csv before concluding the cause.')
collect()
with zipfile.ZipFile(DEST) as z:
 m=json.loads(z.read('manifest.json'));assert all(hashlib.sha256(z.read(n)).hexdigest()==h for n,h in m['files'].items())
print('ARCHIVE',DEST,'SHA256',hashlib.sha256(DEST.read_bytes()).hexdigest())
from google.colab import files
files.download(str(DEST))
